# AI-Generated Image Detector

Upload an image, get a calibrated probability that it is AI-generated.

A two-branch detector: a **frozen CLIP embedding** for semantics plus
**hand-designed frequency features** for forensics, mixed by a *degradation-aware
gate* that estimates how damaged the image is and weights the branches
accordingly. The forensic branch is precise on clean images and collapses under
compression; the semantic branch is coarser but survives. Only the small head is
trained — 563,724 parameters on top of a frozen 86M backbone.

Trained on [SID_Set](https://huggingface.co/datasets/saberzl/SID_Set) plus a
hard-negative pass and an external DALL·E 3 / GAN pass (42,220 images).
Held-out accuracy **0.993**, F1 **0.991**, ROC-AUC **1.000**. On the WildFake
transfer benchmark (never trained on): **0.989** AUC on `laion_matched`,
DALL·E 3 recall **0.93**, Midjourney **0.87**. GANs remain the weak spot —
GigaGAN recall 0.04.

Runs on **CPU** at ~8 images/s, so no GPU runtime is needed.
`Runtime → Run all`, then upload images in section 4 — name them
`real_*` / `ai_*` and it also reports accuracy, precision, recall, F1,
ROC-AUC and a confusion matrix for the batch.

## 1. Install dependencies

In [ ]:
# torch/torchvision ship with Colab; these are the rest.
!pip install -q open_clip_torch scipy scikit-learn Pillow

import torch
print('torch', torch.__version__, '|', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. Get the code and the trained weights

The checkpoint (`checkpoints/full.pt`, 2.2 MB) is committed in the repo, so a
single clone brings both.

In [ ]:
import os, sys, pathlib, subprocess

REPO = 'https://github.com/mxhxmza/ai_img_detector.git'
PROJECT = pathlib.Path('/content/ai_img_detector')

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO,
                    str(PROJECT)], check=True)

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
CKPT = PROJECT / 'checkpoints' / 'full.pt'
print('code   :', PROJECT)
print('weights:', CKPT, f'({CKPT.stat().st_size // 1024} KB)' if CKPT.exists() else 'MISSING')

## 3. Load the model

The first run downloads the frozen CLIP ViT-B/16 backbone (~350 MB) through
`open_clip`; it is cached for the rest of the session.

In [ ]:
from src.inference import Scorer

scorer = Scorer(CKPT, device='auto')
p = scorer.params
print(f"device      : {scorer.device}")
print(f"backbone    : {scorer.config['backbone']}")
print(f"temperature : {scorer.temperature:.3f}")
print(f"parameters  : {p['total']:,} total "
      f"({p['trainable']:,} trainable + {p['frozen_backbone']:,} frozen)")

## 4. Score images — and evaluate, if you label them

Run the cell and pick one or more files.

* **Just checking an image?** Upload it. You get the calibrated probability
  that it is AI-generated and a plain-language verdict.
* **Testing on a labelled set?** Put the true class in the file name prefix —
  `real_*` / `human_*` for authentic, `ai_*` / `fake_*` / `synthetic_*` for
  AI-generated. With at least one labelled image the cell also prints
  **accuracy** and a **confusion matrix**; with at least one of each class it
  adds **precision, recall, F1 and ROC-AUC** for your batch.

In [ ]:
import os, numpy as np
from PIL import Image
from IPython.display import display
from google.colab import files

REAL_PREFIXES = ('real', 'human', 'authentic', 'genuine', 'nonai', '0_')
AI_PREFIXES   = ('ai', 'fake', 'synthetic', 'aigc', 'generated', 'gen_', '1_')


def verdict(p):
    if p >= 0.85: return 'Very likely AI-generated'
    if p >= 0.60: return 'Probably AI-generated'
    if p >  0.40: return 'Uncertain'
    if p >  0.15: return 'Probably authentic'
    return 'Very likely authentic'


def true_label(name):
    n = name.lower()
    if n.startswith(REAL_PREFIXES): return 0
    if n.startswith(AI_PREFIXES):   return 1
    return None


uploaded = files.upload()
names  = list(uploaded)
imgs   = [Image.open(n).convert('RGB') for n in names]
probs  = np.array(scorer.score_many(imgs))
preds  = (probs >= 0.5).astype(int)
labels = [true_label(n) for n in names]

print('=' * 60)
for name, img, p, yhat, y in zip(names, imgs, probs, preds, labels):
    thumb = img.copy(); thumb.thumbnail((240, 240)); display(thumb)
    if y is None:
        tag = ''
    else:
        tag = f"   [labelled {'AI' if y else 'real'} — {'HIT' if y == yhat else 'MISS'}]"
    print(f"  {name}")
    print(f"  p(AI-generated) = {p:6.1%}   ->   {verdict(p)}{tag}\n")

# ---- batch metrics on the labelled images ----------------------------------
lab = [(y, yh, pr) for y, yh, pr in zip(labels, preds, probs) if y is not None]
if not lab:
    print('=' * 60)
    print("  No batch metrics: name the files real_* / ai_* (etc.) to label them.")
else:
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                 f1_score, roc_auc_score, confusion_matrix)
    import matplotlib.pyplot as plt

    ys  = np.array([y  for y, _, _ in lab])
    yhs = np.array([yh for _, yh, _ in lab])
    prs = np.array([pr for _, _, pr in lab])
    cm  = confusion_matrix(ys, yhs, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    both = set(ys.tolist()) == {0, 1}

    print('=' * 60)
    print(f"  batch of {len(lab)} labelled image(s)   |   threshold 0.5\n")
    print(f"  accuracy    {accuracy_score(ys, yhs):.3f}")
    if both:
        print(f"  precision   {precision_score(ys, yhs, zero_division=0):.3f}   "
              f"(of images flagged AI, share that really are)")
        print(f"  recall      {recall_score(ys, yhs, zero_division=0):.3f}   "
              f"(of AI images, share caught)")
        print(f"  F1          {f1_score(ys, yhs, zero_division=0):.3f}")
        try:
            print(f"  ROC-AUC     {roc_auc_score(ys, prs):.3f}")
        except ValueError:
            pass
    else:
        print("  (upload at least one image of each class for precision / recall / F1 / AUC)")
    print(f"\n  confusion matrix")
    print(f"                   pred real   pred AI")
    print(f"    true real   {tn:>10}   {fp:>7}")
    print(f"    true AI     {fn:>10}   {tp:>7}")

    fig, ax = plt.subplots(figsize=(3.4, 3.4))
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['real', 'AI'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['real', 'AI'])
    ax.set_xlabel('predicted'); ax.set_ylabel('actual')
    ax.set_title(f'confusion matrix  (n={len(lab)})')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=14,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.tight_layout(); plt.show()

---

### Reading the score

`pred` is a **calibrated** probability in [0, 1] that the image is fully
AI-generated; **0.5** is the classification threshold. Temperature scaling was
fitted on held-out data after model selection, so the number is meant to be read
at face value rather than as a bare logit.

A *tampered* image — a real photograph with an AI-edited region — is treated as
**real**, because it was still taken by a person. Only fully synthetic images are
flagged.

### Held-out results (5,072 images never trained on)

| metric | value |
|---|---|
| Accuracy | 0.993 |
| Precision | 0.990 |
| Recall | 0.992 |
| F1 | 0.991 |
| ROC-AUC | 1.000 |

20 false positives (0.6% of real photos), 16 false negatives (0.8% of AI).

### Known limitations

Two, both measured on the WildFake transfer benchmark:

- **GANs.** Training is diffusion + ProGAN; modern text-to-image GANs
  (GigaGAN) are still missed ~96% of the time. ProGAN did not transfer.
- **Polished real photos.** Adding DALL·E 3 images raised the false-positive
  rate on LAION-style web photography to ~6% (plain photos: under 1%). The
  appropriate deployment is a human-review queue, not automated enforcement,
  and the threshold can be raised if false positives matter more than recall.

Source: [github.com/mxhxmza/ai_img_detector](https://github.com/mxhxmza/ai_img_detector) · MIT